# LLM-Driven Ontology Generation: IAEDU Pipeline Demonstration



This notebook mirrors the standard pipeline demo but uses the IAEDU-backed option in `ChatGpt` for end-to-end testing.



We'll walk through all 4 phases:

1. **Structured Seed** — Generate a multi-level taxonomic skeleton

2. **Structural Validation** — Validate and prune the seed structure

3. **UCB1 Expansion** — Iteratively expand the ontology using a multi-armed bandit

4. **RDF Serialization** — Emit hierarchical RDF triples and visualize


## Setup



Configure the IAEDU-backed LLM client and ontology parameters.


In [1]:
import json
import logging
import os
from ontogen import ChatGpt, Ontology, DEFAULT_LEVEL_SCHEMA

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True,
    )

# IAEDU-backed configuration
DOMAIN = "Star Trek"  # Change this to explore other domains
API_KEY = os.getenv("IAEDU_API_KEY", "")
ENDPOINT = os.getenv(
    "IAEDU_ENDPOINT",
    "https://api.iaedu.pt/agent-chat//api/v1/agent/cmamvd3n40000c801qeacoad2/stream",
)

CHANNEL_ID = os.getenv("IAEDU_CHANNEL_ID", "cmj1i57292iz9lq01goukbuuv")
PROVIDER = "iaedu"
TIMEOUT = 60
SEED_SIZE = 3  # Number of top-level classes to seed
MAX_ITERATIONS = 60  # Max expansion iterations
CANDIDATES_PER_ITERATION = 3  # Number of candidate relationships to evaluate per iteration
EXPLORATION_CONSTANT = 0.15  # UCB1 exploration parameter
SIMILARITY_THRESHOLD = 70  # Minimum similarity (0-100) to accept relationships
MAX_WORKERS = 2  # IAEDU is sensitive to request bursts during similarity batches
MIN_REQUEST_INTERVAL_SECONDS = 1.0  # Minimum delay between request start times
MAX_CONCURRENT_REQUESTS = 2  # Serialize IAEDU requests to avoid 429 bursts
CLASS_DISCOVERTY_INTERVAL = 50  # Perform class discovery every N iterations to find new candidate classes

print(f"Domain: {DOMAIN}")
print(f"Provider: {PROVIDER}")
print(f"API Key configured: {bool(API_KEY)}")
print(f"Endpoint: {ENDPOINT}")
print(f"Channel ID: {CHANNEL_ID}")
print(f"Ontology workers: {MAX_WORKERS}")
print(f"Request interval seconds: {MIN_REQUEST_INTERVAL_SECONDS}")
print(f"Max concurrent requests: {MAX_CONCURRENT_REQUESTS}")
print(f"Default level schema: {[level.name for level in DEFAULT_LEVEL_SCHEMA]}")


Domain: Star Trek
Provider: iaedu
API Key configured: True
Endpoint: https://api.iaedu.pt/agent-chat//api/v1/agent/cmamvd3n40000c801qeacoad2/stream
Channel ID: cmj1i57292iz9lq01goukbuuv
Ontology workers: 2
Request interval seconds: 1.0
Max concurrent requests: 2
Default level schema: ['class', 'subclass', 'instance']


In [2]:
# Initialize IAEDU-backed LLM client and ontology

agent = ChatGpt(
    api_key=API_KEY,
    provider=PROVIDER,
    endpoint=ENDPOINT,
    channel_id=CHANNEL_ID,
    timeout=TIMEOUT,
    min_request_interval_seconds=MIN_REQUEST_INTERVAL_SECONDS,
    max_concurrent_requests=MAX_CONCURRENT_REQUESTS,
)

ontology = Ontology(
    domain=DOMAIN,
    agent=agent,
    level_schema=DEFAULT_LEVEL_SCHEMA,
    exploration_constant=EXPLORATION_CONSTANT,
    candidates_per_iteration=CANDIDATES_PER_ITERATION,
    max_iterations=MAX_ITERATIONS,
    similarity_threshold=SIMILARITY_THRESHOLD,
    max_workers=MAX_WORKERS,
    class_discovery_interval=CLASS_DISCOVERTY_INTERVAL,
)

print(f"Ontology initialized for domain: {ontology.domain}")
print(f"LLM provider: {agent.provider}")
print(f"Request policy: {agent.describe_request_policy()}")

2026-04-12 23:29:11,423 | INFO | ontogen.llm_client | Initialized iaedu client (provider=iaedu, max_concurrent_requests=2, min_request_interval_seconds=1.00)


Ontology initialized for domain: Star Trek
LLM provider: iaedu
Request policy: provider=iaedu, max_concurrent_requests=2, min_request_interval_seconds=1.00


## Phase 1: Structured Seed

Generate a multi-level taxonomic skeleton from the domain using the LLM.
The seed captures the initial structure with classes, subclasses, and instances.

In [3]:
# Generate structured seed taxonomy
print(f"Generating {SEED_SIZE}-level taxonomic skeleton for '{DOMAIN}'...")
seed = ontology.generate_initial_terms(num_classes=SEED_SIZE)

if seed:
    print(f"\nSeed domain: {seed.get('domain')}")
    print(f"Number of top-level classes: {len(seed.get('taxonomy', []))}")
    print("\nRaw seed taxonomy (JSON):")
    print(json.dumps(seed, indent=2))

else:
    print("ERROR: Seed generation failed. Check IAEDU credentials and try again.")


Generating 3-level taxonomic skeleton for 'Star Trek'...


2026-04-12 23:29:16,836 | INFO | ontogen.llm_client | [IAEDU #1] 'iaedu-chat' completed in 5.39s (slot_wait=0.00s, throttle_wait=0.00s, events=1009, output_chars=5021)
2026-04-12 23:29:16,837 | INFO | ontogen.ontology | Parsed seed successfully: domain=Star Trek, taxonomy count=3



Seed domain: Star Trek
Number of top-level classes: 3

Raw seed taxonomy (JSON):
{
  "domain": "Star Trek",
  "taxonomy": [
    {
      "class": "Species",
      "description": "Different sentient and non-sentient life forms in the Star Trek universe.",
      "subclasses": [
        {
          "class": "Humanoids",
          "description": "Species with human-like physical characteristics.",
          "instances": [
            {
              "term": "Vulcans",
              "description": "A logical and emotion-suppressing species from the planet Vulcan."
            },
            {
              "term": "Klingons",
              "description": "A warrior species known for their honor and combat skills."
            },
            {
              "term": "Betazoids",
              "description": "A telepathic species known for their empathic abilities."
            }
          ]
        },
        {
          "class": "Non-Humanoids",
          "description": "Species with non-hum

## Phase 2: Structural Validation

Convert the seed to a graph and validate its structure using pairwise LLM similarity checks.
This phase prunes weak edges and identifies any orphaned nodes.

In [4]:
# Create DiGraph from seed
print("Converting seed taxonomy to internal graph...")
ontology.create_seed_ontology()

if ontology.ontology_graph.number_of_nodes() > 0:
    print(f"Graph created: {ontology.ontology_graph.number_of_nodes()} nodes, {ontology.ontology_graph.number_of_edges()} edges")
    print(f"\nNodes by level:")
    for level in DEFAULT_LEVEL_SCHEMA:
        nodes_at_level = [n for n, d in ontology.ontology_graph.nodes(data=True) if d.get('level') == level.name]
        print(f"  {level.name}: {len(nodes_at_level)} nodes ({', '.join(nodes_at_level[:3])}{'...' if len(nodes_at_level) > 3 else ''})")
else:
    print("ERROR: Graph creation failed.")

2026-04-12 23:29:16,852 | INFO | ontogen.ontology | Created seed ontology with 31 nodes


Converting seed taxonomy to internal graph...
Graph created: 31 nodes, 28 edges

Nodes by level:
  class: 3 nodes (Species, Starships, Organizations)
  subclass: 9 nodes (Humanoids, Non-Humanoids, Artificial Lifeforms...)
  instance: 19 nodes (Vulcans, Klingons, Betazoids...)


In [5]:
# Validate structure: check parent-child relationships with timed similarity batches
print("\nValidating structure with pairwise similarity checks...")
validation_summary = ontology.validate_structure()

print("\nValidation Summary:")
for key, value in validation_summary.items():
    print(f"  {key}: {value}")

print(f"\nGraph after validation: {ontology.ontology_graph.number_of_nodes()} nodes, {ontology.ontology_graph.number_of_edges()} edges")

2026-04-12 23:29:16,865 | INFO | ontogen.ontology | Generated 28 parent-child validation pairs
2026-04-12 23:29:16,866 | INFO | ontogen.ontology | [Phase 3 validation] Starting similarity batch: total_pairs=28, uncached_pairs=28, workers=2, provider=iaedu, max_concurrent_requests=2, min_request_interval_seconds=1.00
2026-04-12 23:29:16,868 | INFO | ontogen.ontology | [Phase 3 validation] Cache miss: evaluating similarity(Species, Humanoids)
2026-04-12 23:29:16,873 | INFO | ontogen.ontology | [Phase 3 validation] Cache miss: evaluating similarity(Species, Non-Humanoids)
2026-04-12 23:29:16,875 | INFO | ontogen.llm_client | [IAEDU #3] Throttling 'Phase 3 validation' for 0.99s



Validating structure with pairwise similarity checks...


2026-04-12 23:29:17,851 | INFO | ontogen.llm_client | [IAEDU #2] 'Phase 3 validation' completed in 0.98s (slot_wait=0.00s, throttle_wait=0.00s, events=66, output_chars=248)
2026-04-12 23:29:17,853 | INFO | ontogen.ontology | [Phase 3 validation] Similarity(Species, Humanoids) = 75.000000
2026-04-12 23:29:17,855 | INFO | ontogen.ontology | [Phase 3 validation] Cache miss: evaluating similarity(Species, Artificial Lifeforms)
2026-04-12 23:29:17,873 | INFO | ontogen.llm_client | [IAEDU #4] Throttling 'Phase 3 validation' for 1.00s
2026-04-12 23:29:19,059 | INFO | ontogen.llm_client | [IAEDU #3] 'Phase 3 validation' completed in 1.19s (slot_wait=0.00s, throttle_wait=0.99s, events=68, output_chars=241)
2026-04-12 23:29:19,060 | INFO | ontogen.ontology | [Phase 3 validation] Similarity(Species, Non-Humanoids) = 85.000000
2026-04-12 23:29:19,061 | INFO | ontogen.ontology | [Phase 3 validation] Cache miss: evaluating similarity(Humanoids, Vulcans)
2026-04-12 23:29:19,062 | INFO | ontogen.llm_c


Validation Summary:
  edges_pruned: 2
  orphaned_nodes: 1

Graph after validation: 31 nodes, 26 edges


## Phase 3: UCB1 Expansion

Iteratively expand the ontology using a multi-armed bandit (UCB1) strategy.
For each iteration, we select the most promising expandable node and generate new candidates.

In [6]:
# Run full generation pipeline (seed + validate + expand loop + serialize)
print(f"Running full pipeline with max_iterations={MAX_ITERATIONS}...")
print(f"Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"Exploration constant (UCB1): {EXPLORATION_CONSTANT}")
print(f"IAEDU request policy: {agent.describe_request_policy()}")
print(f"Ontology worker count: {ontology.max_workers}\n")

ontology.generate_ontology()

# Post-run audit
print(ontology.history.summary())      # formatted summary
df = ontology.history.to_dataframe()   # pandas DataFrame
ontology.plot_convergence()            # matplotlib convergence charts

print(f"\nPipeline complete!")
print(f"Final graph: {ontology.ontology_graph.number_of_nodes()} nodes, {ontology.ontology_graph.number_of_edges()} edges")

2026-04-12 23:29:47,536 | INFO | ontogen.ontology | Phase 1: Generating seed from domain Star Trek
2026-04-12 23:29:47,537 | INFO | ontogen.llm_client | [IAEDU #30] Throttling 'iaedu-chat' for 0.13s


Running full pipeline with max_iterations=60...
Similarity threshold: 70
Exploration constant (UCB1): 0.15
IAEDU request policy: provider=iaedu, max_concurrent_requests=2, min_request_interval_seconds=1.00
Ontology worker count: 2


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Phase 1: Seed Generation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


2026-04-12 23:29:53,442 | INFO | ontogen.llm_client | [IAEDU #30] 'iaedu-chat' completed in 5.77s (slot_wait=0.00s, throttle_wait=0.13s, events=1149, output_chars=5720)
2026-04-12 23:29:53,443 | INFO | ontogen.ontology | Parsed seed successfully: domain=Star Trek, taxonomy count=5
2026-04-12 23:29:53,444 | INFO | ontogen.ontology | Seed generated: 5 top-level classes
2026-04-12 23:29:53,445 | INFO | ontogen.ontology | Phase 2: Converting seed to ontology graph
2026-04-12 23:29:53,446 | INFO | ontogen.ontology | Created seed ontology with 36 nodes
2026-04-12 23:29:53,446 | INFO | ontogen.ontology | Ontology graph created: 36 nodes, 31 edges
2026-04-12 23:29:53,447 | INFO | ontogen.ontology | Phase 3: Validating structure and pruning weak edges
2026-04-12 23:29:53,448 | INFO | ontogen.ontology | Generated 31 parent-child validation pairs
2026-04-12 23:29:53,448 | INFO | ontogen.ontology | [Phase 3 validation] Starting similarity batch: total_pairs=31, uncached_pairs=17, workers=2, provid

  ✓ Generated 5 top-level classes (5.9s)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Phase 2: Graph Construction
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✓ Created graph: 36 nodes, 31 edges (0.0s)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Phase 3: Structural Validation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


2026-04-12 23:29:54,230 | INFO | ontogen.llm_client | [IAEDU #31] 'Phase 3 validation' completed in 0.78s (slot_wait=0.00s, throttle_wait=0.00s, events=68, output_chars=259)
2026-04-12 23:29:54,231 | INFO | ontogen.ontology | [Phase 3 validation] Similarity(Federation Starships, USS Enterprise) = 85.000000
2026-04-12 23:29:54,232 | INFO | ontogen.ontology | [Phase 3 validation] Cache miss: evaluating similarity(Planets, Federation Planets)
2026-04-12 23:29:54,455 | INFO | ontogen.llm_client | [IAEDU #33] Throttling 'Phase 3 validation' for 1.00s
2026-04-12 23:29:55,415 | INFO | ontogen.llm_client | [IAEDU #32] 'Phase 3 validation' completed in 0.96s (slot_wait=0.00s, throttle_wait=1.00s, events=67, output_chars=221)
2026-04-12 23:29:55,417 | INFO | ontogen.ontology | [Phase 3 validation] Similarity(Klingon Starships, Negh'Var) = 85.000000
2026-04-12 23:29:55,418 | INFO | ontogen.ontology | [Phase 3 validation] Cache miss: evaluating similarity(Planets, Non-Federation Planets)
2026-04-1

...

    37  Dominion Battlecru..     3     3   100%    0.917    138    158  


2026-04-12 23:42:28,095 | INFO | ontogen.llm_client | [IAEDU #606] 'iaedu-chat' completed in 1.13s (slot_wait=0.00s, throttle_wait=0.00s, events=92, output_chars=399)
2026-04-12 23:42:28,096 | INFO | ontogen.ontology | Parsed 3 candidates for node 'Federation Starships'
2026-04-12 23:42:28,098 | INFO | ontogen.ontology | Generated 3 candidates for 'Federation Starships'
2026-04-12 23:42:28,099 | INFO | ontogen.ontology | Validating 3 candidates for parent 'Federation Starships' (threshold=70.0%)
2026-04-12 23:42:28,099 | INFO | ontogen.ontology | [Phase 4 candidate validation for Federation Starships] Starting similarity batch: total_pairs=3, uncached_pairs=3, workers=2, provider=iaedu, max_concurrent_requests=2, min_request_interval_seconds=1.00
2026-04-12 23:42:28,101 | INFO | ontogen.ontology | [Phase 4 candidate validation for Federation Starships] Cache miss: evaluating similarity(Federation Starships, USS Titan)
2026-04-12 23:42:28,102 | INFO | ontogen.ontology | [Phase 4 candida

    38  Federation Starships     3     3   100%    0.850    141    161  


2026-04-12 23:42:45,728 | INFO | ontogen.llm_client | [IAEDU #622] 'iaedu-chat' completed in 1.40s (slot_wait=0.00s, throttle_wait=0.00s, events=102, output_chars=427)
2026-04-12 23:42:45,730 | INFO | ontogen.ontology | Parsed 3 candidates for node 'Klingon Starships'
2026-04-12 23:42:45,730 | INFO | ontogen.ontology | Generated 3 candidates for 'Klingon Starships'
2026-04-12 23:42:45,731 | INFO | ontogen.ontology | Validating 3 candidates for parent 'Klingon Starships' (threshold=70.0%)
2026-04-12 23:42:45,732 | INFO | ontogen.ontology | [Phase 4 candidate validation for Klingon Starships] Starting similarity batch: total_pairs=3, uncached_pairs=3, workers=2, provider=iaedu, max_concurrent_requests=2, min_request_interval_seconds=1.00
2026-04-12 23:42:45,734 | INFO | ontogen.ontology | [Phase 4 candidate validation for Klingon Starships] Cache miss: evaluating similarity(Klingon Starships, Qugh-class)
2026-04-12 23:42:45,737 | INFO | ontogen.ontology | [Phase 4 candidate validation fo

KeyboardInterrupt: 

In [ ]:
# Display expansion statistics
print("\nExpansion Statistics:")
print(f"Total nodes in final ontology: {ontology.ontology_graph.number_of_nodes()}")
print(f"Total edges in final ontology: {ontology.ontology_graph.number_of_edges()}")

# Count nodes by level
print(f"\nNodes by level:")
for level in DEFAULT_LEVEL_SCHEMA:
    nodes_at_level = [n for n, d in ontology.ontology_graph.nodes(data=True) if d.get('level') == level.name]
    print(f"  {level.name}: {len(nodes_at_level)} nodes")

# Show UCB1 bandit statistics
print(f"\nUCB1 Bandit Statistics (node selection):")
if hasattr(ontology, 'bandit_stats') and ontology.bandit_stats:
    for node, stats in sorted(ontology.bandit_stats.items(), key=lambda x: x[1]['n_visits'], reverse=True)[:5]:
        visits = stats.get('n_visits', 0)
        reward = stats.get('total_reward', 0.0)
        mean_reward = reward / visits if visits > 0 else 0
        print(f"  {node}: {visits} visits, avg reward = {mean_reward:.2f}")
else:
    print("  (Bandit statistics not available)")

## Phase 4: RDF Serialization

Convert the expanded ontology to RDF triples and serialize to standard formats (Turtle, XML, JSON-LD).

In [ ]:
# Build RDF graph from the ontology
print("Building RDF graph from ontology...")
ontology.build_ontology()

if ontology.rdf is not None:
    num_triples = len(ontology.rdf)
    print(f"RDF graph built: {num_triples} triples")
else:
    print("ERROR: RDF graph construction failed.")

In [ ]:
# Serialize to Turtle format
print("Serializing ontology to Turtle format...")
turtle_output = ontology.serialize_ontology(format="turtle")

# Display first 50 lines of Turtle output
print("\nTurtle Output (first 50 lines):")
lines = turtle_output.split('\n')
for line in lines[:50]:
    print(line)

if len(lines) > 50:
    print(f"... ({len(lines) - 50} more lines)")

In [ ]:
# Save to file
output_path = "../output/ontology_iaedu.ttl"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w') as f:
    f.write(turtle_output)

print(f"Ontology saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path)} bytes")

## Visualization

Visualize the ontology structure both as an RDF graph and an internal DiGraph with level-based coloring.

In [ ]:
# Visualize the RDF graph
print("Rendering RDF graph (via rdf2dot/graphviz)...")
try:
    ontology.visualize()
except Exception as e:
    print(f"Note: RDF visualization requires graphviz. Error: {e}")

In [ ]:
# Interactive full-page HTML graph (pyvis)

print("Rendering interactive HTML graph...")
html_path = ontology.visualize_interactive(output_path="ontology_iaedu.html")
print(f"Interactive graph saved to: {html_path}")
print("Open the HTML file in a browser for full-page pan-and-zoom exploration.")


In [ ]:
# Visualize the internal DiGraph with level-based node coloring
print("Rendering internal DiGraph with level-based coloring...")
try:
    ontology.visualize_graph()
except Exception as e:
    print(f"Note: DiGraph visualization requires matplotlib. Error: {e}")

In [ ]:
ontology.seed.get("taxonomy", [])

In [ ]:
ontology.history.config


